# Estimación de condición corporal: YOLO y EfficientNet-B0

En este notebook construiremos **paso a paso** un modelo que recibe una fotografía de una vaca y estima su **Body Condition Score (BCS)** como un número continuo entre 1 y 5.

Primero, YOLO localizará la vaca (`cow`) y su grupa (`ass`, nombre utilizado en las anotaciones). Después, EfficientNet-B0 analizará el recorte de la grupa y producirá un valor de BCS. Por ejemplo, una salida podría ser 2.73; no estamos clasificando fotografías en categorías.

El recorrido es: **conocer los datos → comprobar las particiones → preparar YOLO → entrenar el detector → recortar grupas → entrenar el regresor → evaluar → predecir una foto nueva**.

Todo el código se ejecuta aquí, en orden, de arriba abajo. No se crean archivos Python ni se importan módulos del proyecto. Las funciones pequeñas se definen junto al paso que las necesita.

Esta versión reorganiza el experimento con fines didácticos. Las salidas anteriores se han limpiado: las tablas y métricas aparecerán al ejecutarlo. Conserva por separado los outputs de la versión modular anterior; sus mecanismos de reanudación no son compatibles con esta versión.

## 1. Preparar Kaggle

1. Adjunta el dataset con las carpetas `train`, `valid` y `test`; cada una debe contener sus imágenes y `_annotations.coco.json`.
2. Activa una GPU en la configuración del notebook. Para descargar paquetes y pesos iniciales necesitarás acceso a Internet, o adjuntarlos previamente.
3. Ejecuta las celdas en orden. Los entrenamientos usan 80 épocas para YOLO, 60 para la cabeza de regresión y 80 de ajuste fino. No hay parada temprana por estancamiento en ninguno de los dos modelos; se conserva el mejor checkpoint de validación aunque el entrenamiento continúe.

Esta versión usa llamadas directas a `train` y `fit`, y checkpoints convencionales. No incluye el control automático del presupuesto de sesión de la versión modular. Puedes guardar los outputs y continuar desde los checkpoints como se explica al final.

In [ ]:
# Instalamos únicamente Ultralytics si falta; mantenemos los frameworks de Kaggle.
import importlib.util
import importlib.metadata
import subprocess
import sys

ULTRALYTICS_VERSION = "8.4.0"
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        f"ultralytics=={ULTRALYTICS_VERSION}"
    ])
else:
    version = importlib.metadata.version("ultralytics")
    if version != ULTRALYTICS_VERSION:
        raise RuntimeError(
            f"Ultralytics instalado: {version}. Este experimento utiliza "
            f"{ULTRALYTICS_VERSION}; prepara esa versión antes de continuar."
        )

### Importaciones y parámetros

Reunimos las rutas y los hiperparámetros para poder ver qué se usará en cada paso. `SALIDA` es una carpeta nueva para esta versión: así no se mezclan sus resultados con los anteriores. Para cambiar los datos o hiperparámetros, utiliza otro `EXPERIMENTO`.

Dejamos TensorFlow para después del detector y liberamos YOLO antes de entrenar EfficientNet. Al compartir ambos frameworks un kernel, los lotes pequeños ayudan a limitar el uso de memoria.

### Experimento v2: regresión sin parada temprana

En el notebook entrenado v1, la cabeza ejecutó 9 épocas (mejor MAE 0.2613 en la primera) y el ajuste fino 14 (mejor MAE 0.2273 en la sexta). Esto muestra oscilación de validación y no demuestra que más épocas, por sí solas, mejoren el resultado. Probamos una actualización más gradual y un ajuste más amplio:

| Parámetro | v1 | v2 |
|---|---:|---:|
| Lote del regresor | 4 | 8 |
| Épocas de cabeza | 30 (paró en 9) | 60 completas |
| Tasa inicial de cabeza | 0.001 | 0.0003 |
| Épocas de ajuste fino | 20 (paró en 14) | 80 completas |
| Tasa inicial de ajuste | 0.00001 | 0.000005 |
| Últimas capas ajustables, salvo BatchNormalization | 20 | 40 |
| Parada temprana de regresión | Paciencia 8 | Desactivada |
| Tasa de aprendizaje | Fija por fase | Reducción al estancarse |
| Parada temprana de YOLO | Paciencia 15 | Desactivada (`0`) |

Los valores son una hipótesis de mejora que debemos comprobar en validación. Conservamos MAE, arquitectura, aumentos, dropout 0.2, tamaño 224 y semilla para limitar el número de cambios. Al modificar varios hiperparámetros a la vez, este experimento compara la combinación; no identifica cuál de ellos explica una mejora. Para aislar cada efecto, realiza luego experimentos cambiando uno cada vez.

La referencia tiene solo 36 fotos de validación de 10 animales: pequeñas diferencias pueden ser inestables. Conserva test reservado. El nuevo nombre de experimento evita reutilizar los marcadores de fases completas de v1.

El pipeline v1 obtuvo MAE 0.2926 y cobertura 26/36 (72.2 %); nueve abstenciones se debieron a asociaciones ambiguas. Por tanto, mejorar la regresión no resolverá por sí solo la cobertura del detector y su asociación. Mantenemos estos umbrales para comparar sin cambiar otra parte del método.

Referencias de implementación: [ReduceLROnPlateau de Keras](https://keras.io/api/callbacks/reduce_lr_on_plateau/) y [ajuste fino manteniendo BatchNormalization en inferencia](https://keras.io/guides/transfer_learning/).


In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import gc
import hashlib
import json
import math
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
from IPython.display import display
import torch
from ultralytics import YOLO

SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# None busca automáticamente un único dataset COCO dentro de /kaggle/input.
DATOS = None
EXPERIMENTO = "bcs_secuencial_v2_regresion"
SALIDA = Path("/kaggle/working") / EXPERIMENTO
SALIDA.mkdir(parents=True, exist_ok=True)

PESOS_YOLO = "yolo26n.pt"  # También acepta una ruta a pesos adjuntos.
EPOCAS_YOLO = 80
TAM_YOLO = 640
LOTE_YOLO = 4
PACIENCIA_YOLO = 0  # Ultralytics: 0 desactiva la parada temprana.
# Opcional: carpeta detector de los outputs v1, con weights/best.pt y results.csv.
# Permite comparar regresores usando exactamente el mismo detector ya entrenado.
DETECTOR_PREVIO = None  # "/kaggle/input/<outputs>/bcs_secuencial_v1/detector"

TAM_RECORTE = 224
LOTE_REGRESOR = 8
EPOCAS_CABEZA = 60
EPOCAS_AJUSTE = 80
LR_CABEZA = 3e-4
LR_AJUSTE = 5e-6
CAPAS_AJUSTE = 40
FACTOR_REDUCCION_LR = 0.5
PACIENCIA_LR = 8  # Reduce la tasa; no detiene el entrenamiento.
LR_MINIMA = 1e-7
PESOS_EFFICIENTNET = "imagenet"

CONFIANZA = 0.25
COBERTURA_GRUPA = 0.8
TOLERANCIA_CAJA = 0.02  # Píxeles; admite pequeños errores de redondeo COCO.
PARTICION_EVALUACION = "val"
TEST_FINAL_CONFIRMADO = False
IMAGEN_NUEVA = None

assert torch.cuda.is_available(), "Activa una GPU en Kaggle antes de entrenar."
print("GPU del detector:", torch.cuda.get_device_name(0))
print("Resultados:", SALIDA)

## 2. Localizar y conocer el dataset

COCO guarda por separado las imágenes, las categorías y las cajas. Una caja tiene el formato `[x, y, ancho, alto]`, en píxeles. Mostramos primero las cantidades y categorías originales, antes de transformarlas.

La categoría contenedora de Roboflow puede aparecer en la lista de categorías; solo `cow` y `ass` se consideran objetos entrenables. `valid` se llamará `val` en nuestros resultados, pero conservamos las particiones originales.

In [ ]:
if DATOS is None:
    candidatas = sorted({
        p.parent.parent
        for p in Path("/kaggle/input").rglob("_annotations.coco.json")
        if p.parent.name == "train"
        and (p.parent.parent / "valid/_annotations.coco.json").is_file()
        and (p.parent.parent / "test/_annotations.coco.json").is_file()
    })
    if len(candidatas) != 1:
        raise RuntimeError(f"Especifica DATOS manualmente. Candidatas: {candidatas}")
    DATOS = candidatas[0]
DATOS = Path(DATOS)

coco = {}
for carpeta, particion in [("train", "train"), ("valid", "val"), ("test", "test")]:
    coco[particion] = json.loads((DATOS / carpeta / "_annotations.coco.json").read_text())
    contenido = coco[particion]
    print(particion, "| imágenes:", len(contenido["images"]),
          "| cajas:", len(contenido["annotations"]),
          "| categorías:", [(c["id"], c["name"]) for c in contenido["categories"]])

### Leer identidades, BCS y cajas

Las etiquetas `user_tags` contienen la identidad del animal y un valor como `BCS_2_75`, que convertimos a 2.75. Si una fotografía tiene identidades o etiquetas BCS ambiguas, conserva sus cajas para detección pero queda excluida de regresión. No inferimos etiquetas a partir del nombre del archivo.

Comprobamos dimensiones y cajas, y construimos una tabla con una fila por foto. La huella SHA-256 permite detectar archivos idénticos entre particiones. La lectura de imágenes puede tardar un poco.

In [ ]:
# Esta función solo calcula la huella de un archivo, sin cargarlo entero en memoria.
def huella_archivo(ruta):
    huella = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            huella.update(bloque)
    return huella.hexdigest()

In [ ]:
filas = []
correcciones = []
for carpeta, particion in [("train", "train"), ("valid", "val"), ("test", "test")]:
    contenido = coco[particion]
    categorias = {c["id"]: c["name"] for c in contenido["categories"]}
    imagenes = {im["id"]: im for im in contenido["images"]}
    assert len(categorias) == len(contenido["categories"]), "Categorías duplicadas."
    assert {"cow", "ass"} <= set(categorias.values()), "Faltan clases cow/ass."
    assert len(imagenes) == len(contenido["images"]), "IDs de imagen duplicados."
    assert len({a["id"] for a in contenido["annotations"]}) == len(contenido["annotations"])
    anotaciones = defaultdict(list)
    for anotacion in contenido["annotations"]:
        assert anotacion["image_id"] in imagenes, "Referencia de imagen inválida."
        assert categorias[anotacion["category_id"]] in ("cow", "ass")
        assert anotacion.get("iscrowd", 0) == 0, "Revisar anotación iscrowd."
        anotaciones[anotacion["image_id"]].append(anotacion)

    nombres = set()
    for imagen in contenido["images"]:
        nombre = imagen["file_name"]
        assert Path(nombre).name == nombre and "\\" not in nombre
        assert nombre not in nombres, "Nombre de imagen duplicado."
        nombres.add(nombre)
        ruta = DATOS / carpeta / nombre
        with Image.open(ruta) as foto:
            assert foto.size == (imagen["width"], imagen["height"]), str(ruta)
            foto.verify()
        with Image.open(ruta) as foto:
            foto.convert("RGB").load()

        etiquetas = imagen.get("extra", {}).get("user_tags", [])
        identidades = sorted({str(t) for t in etiquetas if re.fullmatch(r"\d+", str(t))})
        etiquetas_bcs = sorted({str(t) for t in etiquetas if str(t).startswith("BCS_")})
        assert identidades, f"Falta identidad para comprobar la partición: {nombre}"
        bcs = None
        if len(identidades) == len(etiquetas_bcs) == 1:
            coincidencia = re.fullmatch(r"BCS_(\d+)(?:_(\d+))?", etiquetas_bcs[0])
            if coincidencia:
                bcs = float(coincidencia[1] + ("." + coincidencia[2] if coincidencia[2] else ""))
                assert 1 <= bcs <= 5, f"BCS fuera de rango: {nombre}"

        cajas = []
        for anotacion in anotaciones[imagen["id"]]:
            x, y, ancho, alto = anotacion["bbox"]
            assert all(math.isfinite(v) for v in (x, y, ancho, alto))
            assert ancho > 0 and alto > 0
            exceso = max(0, -x, -y, x + ancho - imagen["width"], y + alto - imagen["height"])
            assert exceso <= TOLERANCIA_CAJA + 1e-9, f"Caja fuera de imagen: {nombre}"
            xyxy = [max(0, x), max(0, y), min(imagen["width"], x + ancho),
                    min(imagen["height"], y + alto)]
            assert xyxy[2] > xyxy[0] and xyxy[3] > xyxy[1]
            if exceso > 1e-9:
                correcciones.append({"imagen": f"{carpeta}/{nombre}",
                                     "anotacion": anotacion["id"], "xyxy": xyxy})
            cajas.append({"clase": {"cow": 0, "ass": 1}[categorias[anotacion["category_id"]]],
                          "xyxy": xyxy})
        assert Counter(c["clase"] for c in cajas) == {0: 1, 1: 1}, nombre
        filas.append({"clave": f"{carpeta}/{nombre}", "ruta": str(ruta),
                      "particion": particion, "identidades": identidades,
                      "vaca": identidades[0] if len(identidades) == 1 else None,
                      "bcs": bcs, "sha256": huella_archivo(ruta),
                      "ancho": imagen["width"], "alto": imagen["height"], "cajas": cajas})

datos = pd.DataFrame(filas).sort_values("clave").reset_index(drop=True)
print("Fotos leídas:", len(datos), "| BCS válido:", datos["bcs"].notna().sum())
display(datos[["clave", "particion", "vaca", "bcs"]].head())

## 3. Comprobar la separación por animal

Una vaca no debe aparecer a la vez en entrenamiento y validación o test: de lo contrario podríamos medir reconocimiento del animal en lugar de capacidad de generalización. Verificamos **todas** las identidades, incluso las de fotos excluidas de regresión, y buscamos archivos idénticos entre particiones.

Conservamos `train/valid/test` originales si pasan estas comprobaciones. Las huellas no detectan todas las fotos visualmente parecidas ni errores en las identidades; conviene revisar las anotaciones. Los valores de test no se usarán para elegir hiperparámetros.

In [ ]:
particiones_por_vaca = defaultdict(set)
particiones_por_huella = defaultdict(set)
for fila in filas:
    for identidad in fila["identidades"]:
        particiones_por_vaca[identidad].add(fila["particion"])
    particiones_por_huella[fila["sha256"]].add(fila["particion"])

vacas_repetidas = {v: sorted(p) for v, p in particiones_por_vaca.items() if len(p) > 1}
fotos_repetidas = {h: sorted(p) for h, p in particiones_por_huella.items() if len(p) > 1}
assert not vacas_repetidas, f"Vacas en varias particiones: {vacas_repetidas}"
assert not fotos_repetidas, "Existen archivos idénticos entre particiones."
for particion in ("train", "val", "test"):
    assert datos.loc[datos.particion == particion, "bcs"].notna().any(), particion

resumen = datos.groupby("particion").agg(
    fotos_detector=("clave", "size"), fotos_regresor=("bcs", "count"), vacas=("vaca", "nunique")
)
resumen["excluidas_regresion"] = resumen.fotos_detector - resumen.fotos_regresor
display(resumen)
datos.to_json(SALIDA / "datos.json", orient="records", indent=2)
(SALIDA / "correcciones_cajas.json").write_text(json.dumps(correcciones, indent=2))
datos.loc[datos.bcs.isna(), ["clave", "particion"]].to_csv(SALIDA / "excluidas_bcs.csv", index=False)

### Guardar la configuración del experimento

Guardamos parámetros y huellas de los datos para evitar reutilizar accidentalmente checkpoints con otro conjunto o configuración. Si cambias algo, crea otro `EXPERIMENTO`. Las rutas de entrada pueden cambiar al volver a adjuntar el mismo dataset.

In [ ]:
configuracion = {
    "version": "secuencial_2_regresion", "semilla": SEMILLA,
    "datos": [{k: f[k] for k in ("clave", "particion", "sha256", "bcs", "cajas", "identidades")}
              for f in sorted(filas, key=lambda f: f["clave"])],
    "yolo": {"pesos": PESOS_YOLO, "epocas": EPOCAS_YOLO, "tam": TAM_YOLO,
             "lote": LOTE_YOLO, "paciencia": PACIENCIA_YOLO, "version": ULTRALYTICS_VERSION,
             "detector_previo_sha256": (huella_archivo(Path(DETECTOR_PREVIO) / "weights/best.pt")
                                        if DETECTOR_PREVIO is not None else None)},
    "regresor": {"pesos": PESOS_EFFICIENTNET, "tam": TAM_RECORTE, "lote": LOTE_REGRESOR,
                 "epocas_cabeza": EPOCAS_CABEZA, "epocas_ajuste": EPOCAS_AJUSTE,
                 "lr_cabeza": LR_CABEZA, "lr_ajuste": LR_AJUSTE,
                 "capas_ajuste": CAPAS_AJUSTE, "early_stopping": False,
                 "lr_factor": FACTOR_REDUCCION_LR, "lr_paciencia": PACIENCIA_LR,
                 "lr_minima": LR_MINIMA},
    "inferencia": {"confianza": CONFIANZA, "cobertura": COBERTURA_GRUPA}
}
ruta_configuracion = SALIDA / "configuracion.json"
if ruta_configuracion.exists():
    assert json.loads(ruta_configuracion.read_text()) == configuracion, "Usa otro EXPERIMENTO."
else:
    ruta_configuracion.write_text(json.dumps(configuracion, indent=2, allow_nan=False))

### Visualizar ejemplos de entrenamiento

Antes de entrenar, comprobamos que las cajas realmente rodean la vaca y su grupa. Aquí mostramos solo entrenamiento. El BCS es la etiqueta que aprenderá el regresor; la identidad nunca será una entrada del modelo.

In [ ]:
ejemplos = datos[datos.particion == "train"].head(4)
fig, ejes = plt.subplots(1, len(ejemplos), figsize=(16, 5), squeeze=False)
for eje, (_, fila) in zip(ejes[0], ejemplos.iterrows()):
    with Image.open(fila.ruta) as foto:
        eje.imshow(foto.convert("RGB"))
    for caja in fila.cajas:
        x1, y1, x2, y2 = caja["xyxy"]
        color = "lime" if caja["clase"] == 0 else "orange"
        eje.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2))
        eje.text(x1, y1, ["cow", "ass"][caja["clase"]], color=color)
    eje.set_title(f"BCS: {fila.bcs}")
    eje.axis("off")
plt.tight_layout()
plt.show()

## 4. Preparar las etiquetas para YOLO

YOLO espera una línea por objeto: `clase centro_x centro_y ancho alto`. Las coordenadas se normalizan por el tamaño de la imagen. Utilizamos **0 = vaca** y **1 = grupa**; no confundimos este mapeo con los IDs originales de COCO.

Las etiquetas y las copias de las fotos se guardan en `/kaggle/working`. La carpeta de entrada permanece de solo lectura. Esta exportación ocupa espacio, pero puede trasladarse a otra sesión.

In [ ]:
CARPETA_YOLO = SALIDA / "yolo_datos"
for fila in filas:
    nombre = fila["sha256"][:16] + "_" + Path(fila["ruta"]).name
    destino = CARPETA_YOLO / "images" / fila["particion"] / nombre
    etiqueta = CARPETA_YOLO / "labels" / fila["particion"] / (Path(nombre).stem + ".txt")
    destino.parent.mkdir(parents=True, exist_ok=True)
    etiqueta.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fila["ruta"], destino)
    lineas = []
    for caja in fila["cajas"]:
        x1, y1, x2, y2 = caja["xyxy"]
        cx = (x1 + x2) / (2 * fila["ancho"])
        cy = (y1 + y2) / (2 * fila["alto"])
        ancho = (x2 - x1) / fila["ancho"]
        alto = (y2 - y1) / fila["alto"]
        lineas.append(f"{caja['clase']} {cx:.10f} {cy:.10f} {ancho:.10f} {alto:.10f}")
    etiqueta.write_text("\n".join(lineas) + "\n")

RUTA_YAML = SALIDA / "detector.yaml"
# JSON es compatible con YAML y evita problemas al escribir rutas.
RUTA_YAML.write_text(json.dumps({
    "path": str(CARPETA_YOLO), "train": "images/train", "val": "images/val",
    "test": "images/test", "names": {0: "cow", 1: "ass"}
}, indent=2))
print(RUTA_YAML.read_text())

## 5. Entrenar el detector

Partimos de pesos preentrenados de YOLO26n. Entrenamiento ajusta los pesos y validación permite seleccionar `best.pt`; test permanece reservado. `patience=0` desactiva la parada temprana: un entrenamiento nuevo recorrerá las 80 épocas y guardará el mejor checkpoint, que puede corresponder a una época anterior a la última.

La primera ejecución entrena desde los pesos iniciales. Si ya existen resultados completos, los reutilizamos. Tras una interrupción, `resume=True` permite continuar desde `last.pt` **si el checkpoint conserva el optimizador**. Los checkpoints finales pueden tenerlo eliminado; en ese caso se consideran terminados.

Para comparar únicamente el cambio de regresión, puedes configurar `DETECTOR_PREVIO` con la carpeta del detector de v1 adjunta a Kaggle. En ese caso copiamos sus resultados y utilizamos el mismo `best.pt` sin volver a entrenarlo. Si lo dejas en `None`, YOLO se entrena de nuevo sin parada temprana. No reanudes el `last.pt` final de v1 para intentar completar las épocas: ese checkpoint ya tiene eliminado el optimizador.


In [ ]:
CARPETA_DETECTOR = SALIDA / "detector"
ULTIMO_YOLO = CARPETA_DETECTOR / "weights/last.pt"
MEJOR_YOLO = CARPETA_DETECTOR / "weights/best.pt"

if DETECTOR_PREVIO is not None:
    origen_detector = Path(DETECTOR_PREVIO)
    assert (origen_detector / "weights/best.pt").is_file(), "Falta best.pt del detector previo."
    assert (origen_detector / "results.csv").is_file(), "Falta results.csv del detector previo."
    if not CARPETA_DETECTOR.exists():
        shutil.copytree(origen_detector, CARPETA_DETECTOR)
    assert huella_archivo(MEJOR_YOLO) == huella_archivo(origen_detector / "weights/best.pt"), "Detector previo diferente."
    detector = YOLO(str(MEJOR_YOLO))
    print("Reutilizando el detector v1; este experimento cambia solo la regresión.")
elif ULTIMO_YOLO.exists():
    detector = YOLO(str(ULTIMO_YOLO))
    if detector.ckpt.get("optimizer") is not None:
        detector.train(resume=True, device=0, patience=0)
    else:
        assert MEJOR_YOLO.exists(), "Falta best.pt; revisa los outputs del entrenamiento."
        print("Detector ya finalizado; se utilizará best.pt.")
else:
    detector = YOLO(PESOS_YOLO)
    detector.train(
        data=str(RUTA_YAML), epochs=EPOCAS_YOLO, imgsz=TAM_YOLO,
        batch=LOTE_YOLO, device=0, patience=PACIENCIA_YOLO,
        seed=SEMILLA, workers=2, project=str(SALIDA), name="detector",
        exist_ok=True, save=True, save_period=5
    )
assert MEJOR_YOLO.exists(), "No se encontró el mejor checkpoint del detector."

### Revisar las curvas del detector

Las pérdidas describen el ajuste y mAP resume la calidad de localización y clasificación de las cajas. Una mejora en detección no garantiza una mejora del BCS: más adelante mediremos también el pipeline completo.

In [ ]:
historial_yolo = pd.read_csv(CARPETA_DETECTOR / "results.csv")
historial_yolo.columns = historial_yolo.columns.str.strip()
display(historial_yolo.tail())
curvas = [c for c in historial_yolo if "mAP" in c or c.endswith("box_loss")]
historial_yolo.plot(x="epoch", y=curvas, subplots=True, figsize=(10, 9), grid=True)
plt.tight_layout()
plt.show()

# Quitamos referencias a YOLO antes de inicializar TensorFlow.
del detector
gc.collect()
torch.cuda.empty_cache()

## 6. Construir los recortes de grupa

El regresor aprende con las **cajas anotadas** de entrenamiento. Si utilizáramos las predicciones del detector para etiquetar o seleccionar estas muestras, mezclaríamos errores de dos tareas durante el aprendizaje.

Recortamos la grupa, conservamos su proporción y añadimos relleno negro hasta llegar a 224 × 224. Esta misma función se usará con cajas predichas. Redondeamos hacia fuera para incluir toda la caja.

In [ ]:
def recortar_grupa(imagen, caja):
    x1, y1, x2, y2 = caja
    limites = (max(0, math.floor(x1)), max(0, math.floor(y1)),
               min(imagen.width, math.ceil(x2)), min(imagen.height, math.ceil(y2)))
    assert limites[2] > limites[0] and limites[3] > limites[1], "Recorte vacío."
    region = imagen.convert("RGB").crop(limites)
    return ImageOps.pad(region, (TAM_RECORTE, TAM_RECORTE),
                        method=Image.Resampling.BILINEAR, color=(0, 0, 0))

In [ ]:
muestras = []
for fila in filas:
    if fila["bcs"] is None:
        continue
    caja_grupa = next(c["xyxy"] for c in fila["cajas"] if c["clase"] == 1)
    nombre = hashlib.sha256(fila["clave"].encode()).hexdigest() + ".png"
    destino = SALIDA / "recortes_gt" / fila["particion"] / nombre
    destino.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(fila["ruta"]) as foto:
        recortar_grupa(foto, caja_grupa).save(destino)
    muestras.append({"clave": fila["clave"], "particion": fila["particion"],
                     "bcs": fila["bcs"], "recorte": str(destino)})
muestras = pd.DataFrame(muestras)
muestras.to_csv(SALIDA / "muestras_regresion.csv", index=False)
display(muestras.groupby("particion").agg(fotos=("clave", "size")))

In [ ]:
ejemplos = muestras[muestras.particion == "train"].head(6)
fig, ejes = plt.subplots(1, len(ejemplos), figsize=(15, 3), squeeze=False)
for eje, (_, fila) in zip(ejes[0], ejemplos.iterrows()):
    with Image.open(fila.recorte) as imagen:
        eje.imshow(imagen)
    eje.set_title(f"BCS: {fila.bcs:.2f}")
    eje.axis("off")
plt.tight_layout()
plt.show()

## 7. Preparar TensorFlow y los lotes

Leemos los PNG y los agrupamos en lotes. Solo entrenamiento se baraja; validación y evaluación mantienen su orden. El BCS tiene forma `(1,)` por muestra, igual que la salida de la red.

EfficientNet de Keras incluye su propio escalado: recibe RGB en **0 a 255**, por lo que no dividimos otra vez entre 255. Limitamos TensorFlow a una GPU y activamos crecimiento de memoria antes de crear modelos.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

gpus = tf.config.list_physical_devices("GPU")
assert gpus, "TensorFlow no detecta GPU."
try:
    tf.config.set_visible_devices(gpus[0], "GPU")
    tf.config.experimental.set_memory_growth(gpus[0], True)
except RuntimeError:
    print("GPU ya inicializada; si necesitas cambiar su configuración, reinicia el kernel.")
keras.utils.set_random_seed(SEMILLA)
print("TensorFlow:", tf.__version__)

In [ ]:
# tf.data necesita una función para transformar cada par (ruta, etiqueta).
def leer_recorte(ruta, bcs):
    imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
    imagen = tf.ensure_shape(imagen, (TAM_RECORTE, TAM_RECORTE, 3))
    return tf.cast(imagen, tf.float32), tf.cast(bcs, tf.float32)

conjuntos = {}
for particion in ("train", "val"):
    seleccion = muestras[muestras.particion == particion]
    conjunto = tf.data.Dataset.from_tensor_slices((
        seleccion.recorte.to_numpy(dtype=str),
        seleccion.bcs.to_numpy(dtype=np.float32).reshape(-1, 1)
    ))
    if particion == "train":
        conjunto = conjunto.shuffle(len(seleccion), seed=SEMILLA)
    conjuntos[particion] = (
        conjunto.map(leer_recorte, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(LOTE_REGRESOR).prefetch(tf.data.AUTOTUNE)
    )

imagenes_lote, bcs_lote = next(iter(conjuntos["train"]))
print("Imágenes:", imagenes_lote.shape, "| Etiquetas:", bcs_lote.shape)
print("Rango de píxeles:", float(tf.reduce_min(imagenes_lote)), float(tf.reduce_max(imagenes_lote)))

## 8. Construir EfficientNet-B0 para regresión

Usamos EfficientNet como extractor de características y agregamos una salida **lineal de una neurona**. No hay softmax, redondeo ni pesos de clase.

En la primera fase congelamos EfficientNet y entrenamos la cabeza. Los aumentos de imagen se activan automáticamente solo durante entrenamiento. Llamamos a la base con `training=False` para mantener estables las estadísticas de BatchNormalization, también durante el ajuste fino.

La pérdida MAE es el promedio de `|BCS real − BCS predicho|`. Por ejemplo, predecir 3.0 cuando la etiqueta es 2.75 aporta un error de 0.25.

In [ ]:
entrada = keras.Input(shape=(TAM_RECORTE, TAM_RECORTE, 3), name="rgb_0_255")
aumentos = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEMILLA),
    layers.RandomRotation(0.03, seed=SEMILLA + 1),
    layers.RandomZoom(0.05, seed=SEMILLA + 2)
], name="aumentos_entrenamiento")
base = keras.applications.EfficientNetB0(
    include_top=False, weights=PESOS_EFFICIENTNET,
    input_shape=(TAM_RECORTE, TAM_RECORTE, 3)
)
base.trainable = False
x = base(aumentos(entrada), training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2, seed=SEMILLA + 3)(x)
salida_bcs = layers.Dense(1, activation="linear", dtype="float32", name="bcs")(x)
regresor = keras.Model(entrada, salida_bcs, name="regresor_bcs")
regresor.compile(optimizer=keras.optimizers.Adam(LR_CABEZA), loss="mae", metrics=["mae"])
regresor.summary()

### Guardar el último modelo, el mejor y reducir la tasa de aprendizaje

`best.keras` conserva el mejor MAE de validación y `last.keras` el último modelo completo, incluido su optimizador. Un archivo CSV permite revisar las curvas y conocer las épocas finalizadas. Cada fase tiene su propia carpeta.

No utilizamos `EarlyStopping`: las dos fases recorren todas sus épocas. En su lugar, `ReduceLROnPlateau` divide la tasa de aprendizaje entre dos tras 8 épocas sin mejora de validación, hasta un mínimo de `1e-7`. Esto permite seguir entrenando con pasos más pequeños sin detener el proceso. Conservamos `TerminateOnNaN` para interrumpir un entrenamiento numéricamente inválido.

Si vuelves a ejecutar después de una interrupción, retomamos el último modelo y el número de épocas registrado. El optimizador conserva la tasa guardada, pero el contador de estancamiento del callback comienza de nuevo. Esta versión no recupera transaccionalmente todos los estados; si la sesión termina durante el guardado, comprueba los archivos antes de continuar.

Esta pequeña función reúne los mismos callbacks para las dos fases. Las llamadas a `fit` permanecen explícitas en sus celdas. El reductor se ejecuta antes del guardado para conservar la tasa actualizada en el checkpoint.


In [ ]:
def callbacks_fase(carpeta):
    mejor_previo = None
    if (carpeta / "history.csv").exists():
        mejor_previo = float(pd.read_csv(carpeta / "history.csv").val_mae.min())
    return [
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_mae", mode="min", factor=FACTOR_REDUCCION_LR,
            patience=PACIENCIA_LR, min_delta=1e-4, min_lr=LR_MINIMA, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(str(carpeta / "best.keras"), monitor="val_mae",
                                        mode="min", save_best_only=True,
                                        initial_value_threshold=mejor_previo),
        keras.callbacks.ModelCheckpoint(str(carpeta / "last.keras")),
        keras.callbacks.CSVLogger(str(carpeta / "history.csv"), append=True),
        keras.callbacks.TerminateOnNaN()
    ]


## 9. Entrenar la cabeza de regresión

Validación se utiliza durante `fit` para seleccionar el mejor modelo. Test no participa. Si existe `completa.json`, esta fase ya terminó; si solo existe el último checkpoint, continuamos hasta el objetivo total de épocas.

In [ ]:
CABEZA = SALIDA / "regresor/cabeza"
CABEZA.mkdir(parents=True, exist_ok=True)
if not (CABEZA / "completa.json").exists():
    epoca_inicial = 0
    if (CABEZA / "last.keras").exists():
        regresor = keras.models.load_model(CABEZA / "last.keras")
        historial = pd.read_csv(CABEZA / "history.csv")
        epoca_inicial = int(historial.epoch.max()) + 1
    if epoca_inicial < EPOCAS_CABEZA:
        regresor.fit(
            conjuntos["train"], validation_data=conjuntos["val"],
            initial_epoch=epoca_inicial, epochs=EPOCAS_CABEZA,
            callbacks=callbacks_fase(CABEZA)
        )
    assert (CABEZA / "best.keras").exists()
    registro = pd.read_csv(CABEZA / "history.csv")
    assert np.isfinite(registro[["loss", "val_loss", "mae", "val_mae"]].to_numpy()).all(), "Pérdida no finita: revisa el entrenamiento."
    (CABEZA / "completa.json").write_text(json.dumps({"estado": "completa"}))
else:
    print("Fase de cabeza ya completada.")

## 10. Ajustar las últimas capas

Ahora cargamos la mejor cabeza y permitimos modificar las últimas 40 capas de EfficientNet, manteniendo BatchNormalization congelada. Reducimos la tasa de aprendizaje de `3e-4` a `5e-6` para ajustar gradualmente las características preentrenadas.

Al cambiar qué capas son entrenables hay que recompilar. Al **reanudar** un ajuste ya iniciado cargamos el modelo completo sin recompilar, para conservar su optimizador.

In [ ]:
AJUSTE = SALIDA / "regresor/ajuste_fino"
AJUSTE.mkdir(parents=True, exist_ok=True)
if not (AJUSTE / "completa.json").exists():
    epoca_inicial = 0
    if (AJUSTE / "last.keras").exists():
        regresor = keras.models.load_model(AJUSTE / "last.keras")
        historial = pd.read_csv(AJUSTE / "history.csv")
        epoca_inicial = int(historial.epoch.max()) + 1
    else:
        regresor = keras.models.load_model(CABEZA / "best.keras")
        base = regresor.get_layer("efficientnetb0")
        base.trainable = True
        for indice, capa in enumerate(base.layers):
            capa.trainable = (indice >= len(base.layers) - CAPAS_AJUSTE
                              and not isinstance(capa, layers.BatchNormalization))
        regresor.compile(optimizer=keras.optimizers.Adam(LR_AJUSTE), loss="mae", metrics=["mae"])
    if epoca_inicial < EPOCAS_AJUSTE:
        regresor.fit(
            conjuntos["train"], validation_data=conjuntos["val"],
            initial_epoch=epoca_inicial, epochs=EPOCAS_AJUSTE,
            callbacks=callbacks_fase(AJUSTE)
        )
    assert (AJUSTE / "best.keras").exists()
    registro = pd.read_csv(AJUSTE / "history.csv")
    assert np.isfinite(registro[["loss", "val_loss", "mae", "val_mae"]].to_numpy()).all(), "Pérdida no finita: revisa el entrenamiento."
    (AJUSTE / "completa.json").write_text(json.dumps({"estado": "completa"}))
else:
    print("Ajuste fino ya completado.")

### Comparar las fases usando validación

El ajuste fino puede empeorar el resultado. Por eso elegimos entre las mejores versiones de ambas fases utilizando **solo MAE de validación**, y guardamos la selección para la evaluación posterior.

In [ ]:
historiales = {}
for nombre, carpeta in [("cabeza", CABEZA), ("ajuste_fino", AJUSTE)]:
    historiales[nombre] = pd.read_csv(carpeta / "history.csv")
    historiales[nombre].plot(x="epoch", y=["mae", "val_mae"], title=nombre, grid=True)
    plt.ylabel("Error absoluto medio (BCS)")
    plt.show()

mejores_mae = {nombre: float(h.val_mae.min()) for nombre, h in historiales.items()}
fase_elegida = min(mejores_mae, key=mejores_mae.get)
MEJOR_REGRESOR = (CABEZA if fase_elegida == "cabeza" else AJUSTE) / "best.keras"
regresor = keras.models.load_model(MEJOR_REGRESOR)
print("MAE de validación:", mejores_mae)
print("Fase seleccionada:", fase_elegida)
(SALIDA / "seleccion_regresor.json").write_text(json.dumps({
    "fase": fase_elegida, "mae_validacion": mejores_mae,
    "checkpoint": str(MEJOR_REGRESOR.relative_to(SALIDA))
}, indent=2))

comparacion_v1 = pd.DataFrame([
    {"experimento": "v1 observado", "mae_validacion_gt": 0.2273243069648742},
    {"experimento": EXPERIMENTO, "mae_validacion_gt": mejores_mae[fase_elegida]}
])
display(comparacion_v1)
comparacion_v1.to_csv(SALIDA / "comparacion_regresor_v1.csv", index=False)
for nombre, historial in historiales.items():
    mejor_fila = historial.loc[historial.val_mae.idxmin()]
    print(nombre, "| épocas ejecutadas:", len(historial),
          "| mejor época:", int(mejor_fila.epoch) + 1,
          "| mejor MAE:", float(mejor_fila.val_mae))


## 11. Preparar la evaluación del pipeline

Mediremos tres situaciones sobre las mismas fotos con BCS válido:

- **Base de comparación:** predecir siempre la mediana del BCS de entrenamiento.
- **Regresor con caja anotada:** mide el BCS suponiendo una localización correcta.
- **Pipeline completo:** YOLO localiza y EfficientNet estima; puede abstenerse si no hay una asociación única entre vaca y grupa.

La grupa debe tener su centro dentro de la vaca y al menos el 80 % de su área contenida en ella. Si hay varios pares válidos no elegimos arbitrariamente uno. La confianza de YOLO describe la detección, no la confianza del BCS.

In [ ]:
def asociar_grupa(resultado):
    cajas = []
    for coordenadas, clase in zip(resultado.boxes.xyxy.cpu().tolist(),
                                  resultado.boxes.cls.cpu().tolist()):
        cajas.append({"clase": int(clase), "xyxy": coordenadas})
    pares = []
    for vaca in [c for c in cajas if c["clase"] == 0]:
        a, b, c, d = vaca["xyxy"]
        for grupa in [g for g in cajas if g["clase"] == 1]:
            x1, y1, x2, y2 = grupa["xyxy"]
            area = (x2-x1) * (y2-y1)
            interseccion = max(0, min(c, x2)-max(a, x1)) * max(0, min(d, y2)-max(b, y1))
            centro_dentro = a <= (x1+x2)/2 <= c and b <= (y1+y2)/2 <= d
            if area > 0 and centro_dentro and interseccion/area >= COBERTURA_GRUPA:
                pares.append(grupa["xyxy"])
    if len(pares) == 1:
        return pares[0], "ok"
    return None, "sin_par_valido" if not pares else "asociacion_ambigua"

### Reservar test hasta cerrar las decisiones

Por defecto evaluamos `val`. Para el resultado final cambia arriba a `PARTICION_EVALUACION = "test"` y `TEST_FINAL_CONFIRMADO = True`, una vez fijados modelos y umbrales. Se guarda una huella de la selección para detectar cambios posteriores en este experimento.

La protección no reemplaza la disciplina experimental: una vez observado test, no se debe usar su resultado para seguir ajustando el modelo.

In [ ]:
assert PARTICION_EVALUACION in ("val", "test")
seleccion_final = {
    "detector": huella_archivo(MEJOR_YOLO),
    "regresor": huella_archivo(MEJOR_REGRESOR),
    "configuracion": huella_archivo(ruta_configuracion)
}
if PARTICION_EVALUACION == "test":
    assert TEST_FINAL_CONFIRMADO, "Cierra decisiones en validación antes de abrir test."
    bloqueo = SALIDA / "seleccion_test.json"
    if bloqueo.exists():
        assert json.loads(bloqueo.read_text()) == seleccion_final, "La selección cambió después de abrir test."
    else:
        bloqueo.write_text(json.dumps(seleccion_final, indent=2))

### Evaluar la detección

Precisión, recall y mAP describen las cajas en todas las imágenes de la partición. A continuación liberamos YOLO antes de aplicar el regresor a los recortes, reduciendo su coincidencia en memoria.

In [ ]:
# Liberamos el modelo de Keras antes de volver a cargar YOLO.
del regresor, base, entrada, x, salida_bcs, aumentos
keras.backend.clear_session()
gc.collect()
detector = YOLO(str(MEJOR_YOLO))
metricas_detector = detector.val(
    data=str(RUTA_YAML), split=PARTICION_EVALUACION,
    imgsz=TAM_YOLO, batch=LOTE_YOLO, device=0,
    project=str(SALIDA), name="deteccion_" + PARTICION_EVALUACION, exist_ok=True
)
resultado_detector = {k: float(v) for k, v in metricas_detector.results_dict.items()}
display(pd.Series(resultado_detector, name="detección"))
(SALIDA / f"metricas_detector_{PARTICION_EVALUACION}.json").write_text(
    json.dumps(resultado_detector, indent=2)
)

### Detectar y guardar los recortes predichos

Aquí ejecutamos solamente YOLO. Guardamos la causa de abstención y, cuando hay un par válido, el recorte de la grupa. Después ejecutaremos EfficientNet sobre estos recortes usando exactamente el mismo preprocesamiento que con las cajas anotadas.

In [ ]:
evaluacion = datos[(datos.particion == PARTICION_EVALUACION) & datos.bcs.notna()].copy()
carpeta_predichos = SALIDA / "recortes_predichos" / PARTICION_EVALUACION
carpeta_predichos.mkdir(parents=True, exist_ok=True)
detecciones = []
for fila in evaluacion.itertuples():
    resultado = detector.predict(fila.ruta, conf=CONFIANZA, imgsz=TAM_YOLO, device=0, verbose=False)[0]
    caja, estado = asociar_grupa(resultado)
    ruta_recorte = None
    if caja is not None:
        nombre = hashlib.sha256(fila.clave.encode()).hexdigest() + ".png"
        ruta_recorte = str(carpeta_predichos / nombre)
        with Image.open(fila.ruta) as foto:
            recortar_grupa(foto, caja).save(ruta_recorte)
    detecciones.append({"clave": fila.clave, "estado": estado,
                         "caja_predicha": caja, "recorte_predicho": ruta_recorte})
    del resultado

del detector, metricas_detector
gc.collect()
torch.cuda.empty_cache()
print(Counter(d["estado"] for d in detecciones))

### Predecir BCS y calcular métricas

Además de MAE calculamos RMSE, sesgo medio (`predicho − real`), R² y porcentaje de errores de hasta 0.25 y 0.50 BCS. El modelo base usa la mediana de entrenamiento, porque minimiza el error absoluto para una predicción constante.

Para el pipeline, los errores se calculan **solo donde hay predicción**. Reportamos también cobertura y abstenciones; una cobertura baja puede hacer parecer pequeño el MAE al excluir casos difíciles. Por eso añadimos una comparación sobre las mismas fotos cubiertas por el pipeline. No recortamos ni redondeamos las salidas de regresión para mejorar artificialmente las métricas.

In [ ]:
regresor = keras.models.load_model(MEJOR_REGRESOR)
mediana_entrenamiento = float(muestras.loc[muestras.particion == "train", "bcs"].median())
rutas_gt = muestras.set_index("clave").recorte.to_dict()
resultados_bcs = []
for fila, deteccion in zip(evaluacion.itertuples(), detecciones):
    with Image.open(rutas_gt[fila.clave]) as recorte:
        lote = np.asarray(recorte, dtype=np.float32)[None, ...]
    pred_gt = float(regresor.predict(lote, verbose=0).reshape(-1)[0])
    pred_pipeline = None
    if deteccion["recorte_predicho"] is not None:
        with Image.open(deteccion["recorte_predicho"]) as recorte:
            lote = np.asarray(recorte, dtype=np.float32)[None, ...]
        pred_pipeline = float(regresor.predict(lote, verbose=0).reshape(-1)[0])
    resultados_bcs.append({**deteccion, "real": fila.bcs,
                           "base": mediana_entrenamiento,
                           "regresor_gt": pred_gt, "pipeline": pred_pipeline})
predicciones = pd.DataFrame(resultados_bcs)
predicciones.to_csv(SALIDA / f"predicciones_{PARTICION_EVALUACION}.csv", index=False)

In [ ]:
def medir_bcs(tabla, columna):
    disponibles = tabla[columna].notna()
    real = tabla.loc[disponibles, "real"].to_numpy(dtype=float)
    predicho = tabla.loc[disponibles, columna].to_numpy(dtype=float)
    assert np.isfinite(real).all() and np.isfinite(predicho).all()
    error = predicho - real
    total = len(tabla)
    resultado = {"evaluables": total, "predicciones": len(error),
                 "abstenciones": total - len(error),
                 "cobertura": len(error)/total if total else None,
                 "mae": None, "rmse": None, "sesgo": None, "r2": None,
                 "error_hasta_025": None, "error_hasta_050": None}
    if len(error):
        denominador = np.sum((real - real.mean())**2)
        resultado.update(mae=float(np.abs(error).mean()), rmse=float(np.sqrt(np.mean(error**2))),
                         sesgo=float(error.mean()),
                         r2=float(1 - np.sum(error**2)/denominador) if denominador > 0 else None,
                         error_hasta_025=float(np.mean(np.abs(error) <= 0.25)),
                         error_hasta_050=float(np.mean(np.abs(error) <= 0.50)))
    return resultado

metricas = {columna: medir_bcs(predicciones, columna)
            for columna in ("base", "regresor_gt", "pipeline")}
cubiertas = predicciones[predicciones.pipeline.notna()]
for columna in ("base", "regresor_gt"):
    metricas[columna + "_mismas_fotos"] = medir_bcs(cubiertas, columna)
display(pd.DataFrame(metricas).T)
(SALIDA / f"metricas_bcs_{PARTICION_EVALUACION}.json").write_text(
    json.dumps(metricas, indent=2, allow_nan=False)
)

## 12. Interpretar los resultados

En el gráfico real frente a predicho, la diagonal representa una predicción perfecta. Los puntos alejados indican errores grandes. Mostramos también algunos de los peores casos del pipeline para comprobar visualmente la localización.

Compara el regresor con cajas anotadas y el pipeline sobre las mismas fotos. Si el primero funciona mejor, los recortes predichos pueden estar introduciendo errores. Si ambos fallan de forma parecida, conviene revisar el aprendizaje de BCS y las etiquetas, usando validación para las decisiones.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, columna in zip(ejes, ["regresor_gt", "pipeline"]):
    seleccion = predicciones[predicciones[columna].notna()]
    eje.scatter(seleccion.real, seleccion[columna], alpha=0.65)
    eje.plot([1, 5], [1, 5], "k--")
    eje.set(xlabel="BCS real", ylabel="BCS predicho", title=columna)
    eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

predicciones["error_absoluto"] = (predicciones.pipeline - predicciones.real).abs()
peores = predicciones.dropna(subset=["pipeline"]).nlargest(4, "error_absoluto")
if len(peores):
    fig, ejes = plt.subplots(1, len(peores), figsize=(14, 4), squeeze=False)
    originales = datos.set_index("clave").ruta.to_dict()
    for eje, fila in zip(ejes[0], peores.itertuples()):
        with Image.open(originales[fila.clave]) as foto:
            eje.imshow(foto.convert("RGB"))
        x1, y1, x2, y2 = fila.caja_predicha
        eje.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="orange", linewidth=2))
        eje.set_title(f"Real {fila.real:.2f} | Pred. {fila.pipeline:.2f}")
        eje.axis("off")
    plt.tight_layout()
    plt.show()
display(predicciones.loc[predicciones.pipeline.isna(), ["clave", "estado"]].head(10))

## 13. Predecir una fotografía nueva (opcional)

Configura `IMAGEN_NUEVA` con una imagen adjunta y ejecuta esta celda. Primero detectamos la vaca y la grupa; después estimamos BCS con el recorte. Si no hay asociación única, mostramos una abstención en lugar de inventar un valor.

In [ ]:
if IMAGEN_NUEVA is not None:
    del regresor
    keras.backend.clear_session()
    gc.collect()
    detector = YOLO(str(MEJOR_YOLO))
    resultado = detector.predict(str(IMAGEN_NUEVA), conf=CONFIANZA,
                                 imgsz=TAM_YOLO, device=0, verbose=False)[0]
    caja, estado = asociar_grupa(resultado)
    display(Image.fromarray(resultado.plot()[..., ::-1]))
    del detector, resultado
    gc.collect()
    torch.cuda.empty_cache()
    regresor = keras.models.load_model(MEJOR_REGRESOR)
    prediccion_nueva = {"estado": estado, "bcs": None}
    if caja is not None:
        with Image.open(IMAGEN_NUEVA) as foto:
            recorte = recortar_grupa(foto, caja)
        display(recorte)
        lote = np.asarray(recorte, dtype=np.float32)[None, ...]
        prediccion_nueva["bcs"] = float(regresor.predict(lote, verbose=0).reshape(-1)[0])
    print(prediccion_nueva)
    (SALIDA / "prediccion_nueva.json").write_text(json.dumps(prediccion_nueva, indent=2))
else:
    print("Configura IMAGEN_NUEVA si deseas probar otra fotografía.")

## 14. Guardar y continuar en otra sesión

Guardamos un ZIP con los checkpoints, historiales, configuración, métricas y predicciones. Excluimos las copias preparadas de fotos y recortes para reducir su tamaño: se regeneran desde el dataset original.

Para continuar:

1. Guarda los outputs de Kaggle y conserva el notebook junto con el dataset original.
2. En una sesión nueva adjunta los outputs. Extrae el ZIP en `/kaggle/working` antes de ejecutar las celdas de configuración, o copia allí la carpeta del experimento guardada.
3. Mantén `EXPERIMENTO` y los hiperparámetros. Ejecuta en orden: los datos preparados se regeneran y las fases completadas se reutilizan.

YOLO necesita el optimizador de `last.pt` para reanudar un entrenamiento interrumpido. Keras necesita `last.keras` y `history.csv` de la fase. Si una interrupción ocurrió durante el guardado, revisa los archivos; esta versión no promete recuperación automática de checkpoints incompletos.

El ZIP contiene información del experimento y posiblemente imágenes de los reportes del detector: consérvalo de forma privada. El archivo exportado no incluye este notebook; guárdalo también desde Kaggle.

Para iniciar v2 desde cero, no copies la carpeta `regresor` de v1 a esta salida. Para reutilizar solo el detector, adjunta sus outputs y usa `DETECTOR_PREVIO`; para reanudar v2 sí debes recuperar sus propios checkpoints y su configuración. El nuevo ZIP se llama `bcs_secuencial_v2_regresion_resultados.zip`.


In [ ]:
import zipfile

ruta_zip = SALIDA.parent / (EXPERIMENTO + "_resultados.zip")
carpetas_regenerables = {"yolo_datos", "recortes_gt", "recortes_predichos"}
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as archivo:
    for ruta in sorted(SALIDA.rglob("*")):
        relativa = ruta.relative_to(SALIDA)
        if ruta.is_file() and relativa.parts[0] not in carpetas_regenerables:
            archivo.write(ruta, str(Path(EXPERIMENTO) / relativa))
print("Archivo para descargar:", ruta_zip)
print(f"Tamaño: {ruta_zip.stat().st_size / 1024**2:.1f} MB")